# 03 — Foundation Models Quickstart: CausalPFN, Do-PFN & CausalFM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chris-L6/causalfm-survey/blob/main/notebooks/03_foundation_models_quickstart.ipynb)

**An introduction for practitioners: simulate one dataset, then run all
three causal foundation models on it using each library's own native API.**

This notebook is standalone — it doesn't import this repo's `causal_bench`
wrappers. Every call below is exactly what you'd write reading each model's own
README, so you can lift a cell straight into your own project. Structure:

<!-- | Step | What it does |
|---|---|
| §0 | One-time environment check — **run this first** if you want Do-PFN to work |
| §1 | Simulate one dataset with a known (but normally unobservable) treatment effect |
| §2 | Run **CausalPFN** — `pip install causalpfn`, native `CATEEstimator` / `ATEEstimator` |
| §3 | Run **Do-PFN** — `git clone`, native `DoPFNRegressor` |
| §4 | Run **CausalFM** — `git clone` + checkpoint, native `StandardCATEModel` |
| §5 | Compare all three against ground truth | -->


## Why these aren't just "another sklearn model"

CausalPFN, Do-PFN and CausalFM are **amortized** / **in-context** estimators: a single
transformer is pretrained once (by the model authors, on millions of synthetic
causal-inference problems) and shipped as frozen weights. There is no
per-dataset training loop on your end — `fit()` just packages your data as
"context" for a forward pass.

```python
# Traditional metalearner (e.g. T-learner): trains fresh parameters on YOUR data
model_treated = RandomForest().fit(X[T == 1], Y[T == 1])
model_control = RandomForest().fit(X[T == 0], Y[T == 0])
tau_hat = model_treated.predict(X_test) - model_control.predict(X_test)

# Causal foundation model: weights are already trained; "fit" just stores context
cate_estimator = CATEEstimator(device=device)     # pretrained weights, downloaded once
cate_estimator.fit(X_train, T_train, Y_train)      # NOT gradient descent on your data
tau_hat = cate_estimator.estimate_cate(X_test)     # one forward pass, conditioned on context
```

**Practical upshot:** `fit()` is cheap and the same frozen network is reused
across every dataset you throw at it — that's what "zero-shot" means here. The
three models below differ mainly in *how* you get their weights (PyPI package
vs. `git clone`) and the exact shape of their `fit`/`predict` calls — each
section shows its own, unmodified from the source library.

## 0. One-time environment check — run this cell FIRST

**Only needed if you want Do-PFN to work** (§3). Its model code depends on an
internal PyTorch name (`Optional`, re-exported from `torch.nn.modules.transformer`)
that PyTorch removed in `torch>=2.10` — verified against PyTorch's own source
history (present through `v2.9.0`, gone in `v2.10.0`). This must run *before*
any other cell, because `torch` gets imported by the next section, and a pip
downgrade has no effect on an already-imported module without a restart.

- **On Colab**: this cell installs `torch<2.10` and **restarts the runtime for
  you** (you'll see it disconnect/reconnect — expected). After it reconnects,
  run this cell again — it will print "OK" — then continue through the
  notebook top to bottom as normal.
- **Locally, in this repo's `uv` venv**: `pip` doesn't exist here at all (`No
  module named pip`), so a `!pip install` inside the notebook silently does
  **nothing** — this cell can only detect the problem locally, not fix it. Run
  the fix in a terminal instead:
  ```bash
  uv pip install "torch<2.10"
  ```
  then restart this notebook's kernel and re-run from the top.
- **Not planning to run Do-PFN?** Skip this — CausalPFN and CausalFM both work
  fine on any recent torch.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

def _torch_pre_2_10():
    try:
        import torch
    except ImportError:
        return True  # not installed yet -- nothing to fix here
    major, minor = (int(p) for p in torch.__version__.split("+")[0].split(".")[:2])
    return (major, minor) < (2, 10)

if _torch_pre_2_10():
    print("OK -- torch version is compatible with Do-PFN (or not installed yet).")
elif IN_COLAB:
    import subprocess
    print("torch >= 2.10 detected -- installing torch<2.10 and restarting the runtime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch<2.10"], check=True)
    print("Restarting now. After it reconnects, re-run THIS cell, then continue from the top.")
    os.kill(os.getpid(), 9)  # Colab reconnects automatically with a fresh process
else:
    import torch
    print(f"torch {torch.__version__} is >= 2.10 -- Do-PFN (section 3) will fail to import.")
    print('Fix, in a terminal (not this notebook -- local uv venv has no pip):')
    print('    uv pip install "torch<2.10"')
    print("then restart this notebook's kernel and re-run from the top.")

## 1. Example dataset — a simulated discount-email campaign

Rather than an abstract `X0, X1, ...` matrix, we simulate a small business
scenario in the spirit of Facure's *Causal Inference for the Brave and True*:
an online retailer sends a discount email to a subset of customers and wants
to know its effect on next-month spend.

- **Covariates**: `recency` (days since last purchase, standardized — higher
  means more lapsed), `monetary` (average past order value, standardized),
  `age` (standardized).
- **Treatment** `T`: received the discount email. It's **confounded on
  purpose** — marketing targets loyal, high-spend, recently-active customers,
  so a naive treated-vs-untreated comparison is biased.
- **Outcome** `Y`: next-month spend.
- **Ground truth** (known only because this is simulated, never in real data):
  the email works *better* on lapsed customers and *worse* on older ones —
  `tau(x) = 2.0 + 1.5 * recency - 0.75 * age`, a heterogeneous effect that lets
  us score each model's CATE estimate, not just its ATE.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)
n = 1500

recency = rng.normal(0, 1, n)    # standardized days since last purchase
monetary = rng.normal(0, 1, n)   # standardized average past order value
age = rng.normal(0, 1, n)        # standardized customer age
X = np.column_stack([recency, monetary, age]).astype(np.float32)

# Confounded treatment assignment: loyal, high-spend, recently-active
# customers are more likely to be targeted with the discount email.
propensity = 1 / (1 + np.exp(-(0.8 * monetary - 0.6 * recency)))
T = rng.binomial(1, propensity).astype(np.float32)

# True heterogeneous treatment effect (unobservable outside a simulation).
tau_true = (2.0 + 1.5 * recency - 0.75 * age).astype(np.float32)

# Potential outcomes -> observed outcome.
noise = rng.normal(0, 1.0, n).astype(np.float32)
Y0 = (5.0 + 2.0 * monetary - 0.5 * age + noise).astype(np.float32)
Y1 = Y0 + tau_true
Y = np.where(T == 1, Y1, Y0).astype(np.float32)

X_train, X_test, T_train, T_test, Y_train, Y_test, tau_train, tau_test = train_test_split(
    X, T, Y, tau_true, test_size=0.3, random_state=SEED
)
true_ate = float(tau_true.mean())

print(f"n_train / n_test               : {len(X_train)} / {len(X_test)}")
print(f"Naive treated-vs-control gap   : {Y[T == 1].mean() - Y[T == 0].mean():.3f}  (biased by confounding)")
print(f"True ATE (known only here)     : {true_ate:.3f}")

In [ ]:
# Tiny scoring helpers -- kept inline since this notebook has no other deps.
import os, sys, time

def pehe(tau_hat, tau_ref):
    return float(np.sqrt(np.mean((np.asarray(tau_hat) - np.asarray(tau_ref)) ** 2)))

def ate_abs_error(ate_hat, ate_ref):
    return float(abs(ate_hat - ate_ref))

import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device: {device}")

IN_COLAB = "google.colab" in sys.modules
# NOTE (local `uv` venvs only): `%pip install` / `!pip install` cells below
# silently no-op locally ("No module named pip") -- they only work on Colab,
# which ships pip. Locally, install missing packages with `uv pip install <pkg>`.

results = {}  # model name -> dict(tau_hat, ate_hat, runtime, pehe, ate_abs_error)

## 2. CausalPFN

**Install:** `pip install causalpfn` — a normal PyPI package. The first call
downloads pretrained weights from the Hugging Face Hub (a few hundred MB), so
it needs internet access once; after that they're cached locally.

**Caveat (verified directly, not copied from the package docs):** on Apple
Silicon macOS, CausalPFN **segfaults** — a hard process crash, not a Python
exception `try/except` can catch — on *both* `device="cpu"` and `"mps"`. The
package has no `torch.compile` call; the likely cause is
`F.scaled_dot_product_attention` (`transformer_layer.py`) hitting an unstable
SDPA kernel on macOS's CPU/MPS backends, a known category of PyTorch bug
rather than a hard CUDA requirement. So this cell **proactively skips** on
that specific combination instead of trying and crashing the kernel. On
Colab, this is very likely *not* an issue on either runtime — GPU (CUDA) is
the best-supported path, and CPU (Linux x86_64, mature SDPA kernels) probably
works too, though we haven't verified the Colab-CPU case ourselves.

The call below is CausalPFN's own two-estimator API, unmodified — a
`CATEEstimator` for the per-unit effect and a separate `ATEEstimator` for the
population average:

In [ ]:
%pip install -q causalpfn
import platform

# Verified by direct test: segfaults on Apple Silicon macOS for both CPU and
# MPS (not a Python exception, so we check *before* calling rather than
# wrapping in try/except). Any other platform/device combo is allowed to try.
APPLE_SILICON_MACOS = platform.system() == "Darwin" and platform.machine() == "arm64"

if device != "cuda" and APPLE_SILICON_MACOS:
    print("⚠ Skipping CausalPFN: verified segfault on Apple Silicon macOS "
          "(both CPU and MPS) -- likely an unstable scaled_dot_product_attention "
          "kernel on this platform's backend, not a hard CUDA requirement. "
          "Should be fine on Colab (GPU, and probably CPU too).")
else:
    try:
        from causalpfn import CATEEstimator, ATEEstimator
    except ImportError:
        print("✗ causalpfn not installed -- run the %pip install line above.")
    else:
        t0 = time.time()

        cate_estimator = CATEEstimator(device=device, verbose=False)
        cate_estimator.fit(X_train, T_train, Y_train)
        tau_hat = np.asarray(cate_estimator.estimate_cate(X_test)).reshape(-1)

        ate_estimator = ATEEstimator(device=device, verbose=False)
        ate_estimator.fit(X_train, T_train, Y_train)
        ate_hat = float(np.asarray(ate_estimator.estimate_ate()).reshape(-1)[0])

        runtime = time.time() - t0
        results["CausalPFN"] = dict(
            tau_hat=tau_hat, ate_hat=ate_hat, runtime=runtime,
            pehe=pehe(tau_hat, tau_test), ate_abs_error=ate_abs_error(ate_hat, true_ate),
        )
        print(f"✓ CausalPFN | PEHE={results['CausalPFN']['pehe']:.3f}  "
              f"ATE_hat={ate_hat:.3f}  runtime={runtime:.2f}s")

## 3. Do-PFN

**How to run it — Do-PFN is *not* on PyPI**, and its own `requirements.txt` is
a frozen research/benchmark environment, not a minimal runtime spec — pinning
things like `catboost==1.1.1` (which has no wheel for recent Python and will
fail `pip install -r Do-PFN/requirements.txt` outright) purely for baseline
comparisons that `DoPFNRegressor` itself never imports. This cell installs only
what the regressor actually needs beyond what's already installed above
(`torch`, `numpy`, `scipy`, `pandas`, `scikit-learn`): `networkx`, `tqdm`,
`einops`.

Three gotchas verified by tracing Do-PFN's source, none of which are obvious
from its README:

1. **Import path.** The correct, current import is
   `from scripts.transformer_prediction_interface import DoPFNRegressor` — not
   `from dopfn import ...` or `from model.dopfn import ...`.
2. **Treatment must be column 0**, not appended at the end — `predict_cid`
   does `X[:, 0] = t` internally.
3. **`DoPFNRegressor()` loads its checkpoint via a path relative to the repo
   root** (`artifacts/...`), on *both* construction and first `fit()` call —
   so the working directory must be the `Do-PFN/` folder for that whole block,
   not just on `sys.path`.

Do-PFN exposes a dedicated `predict_cate(X)` method (internally: predict at
`do(T=1)` minus `do(T=1)` — no need to call `predict_full` twice by hand), but
its internal input validation requires a `torch.Tensor`, not a plain numpy
array.

**One more, torch-version-specific:** Do-PFN's `model/layer.py` imports
`Optional` (among other names) from `torch.nn.modules.transformer` — an
internal re-export PyTorch dropped in **v2.10.0** (verified directly against
PyTorch's source: present in the v2.9.0 tag, gone in v2.10.0). If you skipped
the §0 environment check above and hit `cannot import name 'Optional' from
'torch.nn.modules.transformer'` here, go run that cell first.

In [ ]:
import subprocess

DOPFN_DIR = "Do-PFN"
DOPFN_URL = "https://github.com/jr2021/Do-PFN.git"

if not os.path.exists(DOPFN_DIR):
    print(f"Cloning {DOPFN_URL} ...")
    subprocess.run(["git", "clone", DOPFN_URL], check=True)
sys.path.insert(0, os.path.abspath(DOPFN_DIR))

# NOT `pip install -r Do-PFN/requirements.txt` -- see markdown above.
if IN_COLAB:
    get_ipython().system("pip install -q networkx tqdm einops")
# Locally: uv pip install networkx tqdm einops

try:
    from scripts.transformer_prediction_interface import DoPFNRegressor
except ImportError as e:
    DoPFNRegressor = None
    if "torch.nn.modules.transformer" in str(e):
        # PyTorch dropped this internal re-export in v2.10.0 (present through
        # v2.9.0, verified against PyTorch's own source) -- Do-PFN's model
        # code still relies on it.
        print(f"✗ Do-PFN not importable: incompatible torch version ({e})\n"
              "  Run the '0. One-time environment check' cell at the top of "
              "this notebook, then follow its instructions and re-run from the top.")
    else:
        print(f"✗ Do-PFN not importable: {e}")

if DoPFNRegressor is not None:
    t0 = time.time()

    # Treatment in COLUMN 0 -- see gotcha #2 above.
    X_full_train = np.concatenate([T_train.reshape(-1, 1), X_train], axis=1)
    X_full_test = np.concatenate(
        [np.zeros((len(X_test), 1), dtype=np.float32), X_test], axis=1
    )  # column 0 here is a placeholder; predict_cate overwrites it internally

    _cwd = os.getcwd()
    os.chdir(DOPFN_DIR)  # relative checkpoint path -- see gotcha #3 above
    try:
        dopfn = DoPFNRegressor()
        dopfn.fit(X_full_train, Y_train)
        tau_hat = np.asarray(dopfn.predict_cate(torch.as_tensor(X_full_test))).reshape(-1)
    finally:
        os.chdir(_cwd)

    ate_hat = float(tau_hat.mean())
    runtime = time.time() - t0
    results["Do-PFN"] = dict(
        tau_hat=tau_hat, ate_hat=ate_hat, runtime=runtime,
        pehe=pehe(tau_hat, tau_test), ate_abs_error=ate_abs_error(ate_hat, true_ate),
    )
    print(f"✓ Do-PFN | PEHE={results['Do-PFN']['pehe']:.3f}  "
          f"ATE_hat={ate_hat:.3f}  runtime={runtime:.2f}s")

## 4. CausalFM

**How to run it — also not on PyPI**, and it additionally needs a pretrained
checkpoint file:

1. `git clone https://github.com/yccm/CausalFM-toolkit.git`
2. install just the extra deps it needs (`einops`, `tabpfn==2.0.9`,
   `tensorboard`) — **not** its bundled `requirements.txt`, which is a frozen
   Linux/CUDA dev snapshot that won't install on macOS or a Colab CPU runtime
3. add the toolkit root to `sys.path`
4. point at the real checkpoint path:
   `checkpoints/checkpoints_standard/best_model.pth` (note: **not**
   `checkpoints/best_model.pth`, despite the toolkit's own README)

One more gotcha worth calling out inline: the toolkit's README quick-start
shows `model.estimate_cate(x_train, a_train, y_train, x_test)` with plain
numpy arrays, but the actual `StandardCATEModel.estimate_cate` requires
`torch.Tensor` inputs with treatment/outcome reshaped to `[N, 1]` — the call
below uses the shapes it actually needs, not the README's simplified one.

**On Colab you'll likely see a harmless pip resolver warning here** —
`tabpfn==2.0.9` pins `huggingface-hub<1`, which conflicts with Colab's
preinstalled `gradio`/`transformers` (they want `huggingface-hub>=1.x`). pip
still installs everything requested and prints the conflict as a warning, not
an error; since this notebook never imports `gradio` or `transformers`, it's
safe to ignore — CausalFM loads and runs normally right after.

In [ ]:
CAUSALFM_DIR = "CausalFM-toolkit"
CAUSALFM_URL = "https://github.com/yccm/CausalFM-toolkit.git"
CAUSALFM_CHECKPOINT = f"{CAUSALFM_DIR}/checkpoints/checkpoints_standard/best_model.pth"

if not os.path.exists(CAUSALFM_DIR):
    print(f"Cloning {CAUSALFM_URL} ...")
    subprocess.run(["git", "clone", CAUSALFM_URL], check=True)
sys.path.insert(0, os.path.abspath(CAUSALFM_DIR))

if IN_COLAB:
    get_ipython().system('pip install -q einops "tabpfn==2.0.9" tensorboard')
# Locally: uv pip install einops "tabpfn==2.0.9" tensorboard

try:
    from causalfm.models import StandardCATEModel
except ImportError:
    StandardCATEModel = None

if StandardCATEModel is None:
    print("✗ causalfm not importable -- see the install comments above.")
elif not os.path.exists(CAUSALFM_CHECKPOINT):
    print(f"✗ checkpoint not found at {CAUSALFM_CHECKPOINT}")
else:
    t0 = time.time()
    model = StandardCATEModel.from_pretrained(CAUSALFM_CHECKPOINT)

    X_train_t = torch.as_tensor(X_train, dtype=torch.float32)
    T_train_t = torch.as_tensor(T_train, dtype=torch.float32).reshape(-1, 1)
    Y_train_t = torch.as_tensor(Y_train, dtype=torch.float32).reshape(-1, 1)
    X_test_t = torch.as_tensor(X_test, dtype=torch.float32)

    result = model.estimate_cate(X_train_t, T_train_t, Y_train_t, X_test_t)
    tau_hat = result["cate"].detach().cpu().numpy().reshape(-1)

    ate_hat = float(tau_hat.mean())
    runtime = time.time() - t0
    results["CausalFM"] = dict(
        tau_hat=tau_hat, ate_hat=ate_hat, runtime=runtime,
        pehe=pehe(tau_hat, tau_test), ate_abs_error=ate_abs_error(ate_hat, true_ate),
    )
    print(f"✓ CausalFM | PEHE={results['CausalFM']['pehe']:.3f}  "
          f"ATE_hat={ate_hat:.3f}  runtime={runtime:.2f}s")

## 5. Compare & visualize

Whatever subset of the three models ran successfully in your environment gets
plotted here — left panel: estimated CATE vs. ground truth (perfect
predictions sit on the diagonal); right panel: PEHE per model (lower is
better). Colors are assigned per model, consistently across both panels.

In [ ]:
import matplotlib.pyplot as plt

# Fixed model -> color assignment, consistent across every panel below.
MODEL_COLORS = {"CausalPFN": "#2a78d6", "Do-PFN": "#eb6834", "CausalFM": "#1baf7a"}

if not results:
    print("No foundation model ran successfully in this environment -- "
          "see the ✗/⚠ messages above for what to install.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Left: predicted vs. true CATE, one scatter series per model
    ax = axes[0]
    lo = min(tau_test.min(), *[r["tau_hat"].min() for r in results.values()])
    hi = max(tau_test.max(), *[r["tau_hat"].max() for r in results.values()])
    ax.plot([lo, hi], [lo, hi], color="#8a8a86", linewidth=1.5, linestyle="--", label="perfect (y = x)")
    for name, r in results.items():
        ax.scatter(tau_test, r["tau_hat"], s=14, alpha=0.5,
                    color=MODEL_COLORS[name], label=name)
    ax.set_xlabel("True CATE")
    ax.set_ylabel("Predicted CATE")
    ax.set_title("Predicted vs. true treatment effect")
    ax.legend(frameon=False, fontsize=9)

    # Right: PEHE bar per model (lower = better)
    ax = axes[1]
    names = list(results.keys())
    pehes = [results[n]["pehe"] for n in names]
    ax.bar(names, pehes, color=[MODEL_COLORS[n] for n in names])
    for i, v in enumerate(pehes):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("PEHE (lower is better)")
    ax.set_title("Heterogeneous-effect error, discount-email example")

    plt.tight_layout()
    plt.savefig("foundation_models_quickstart.png", dpi=150)
    plt.show()

    summary = pd.DataFrame({
        name: {"ATE_hat": r["ate_hat"], "True_ATE": true_ate, "PEHE": r["pehe"],
               "ATE_abs_error": r["ate_abs_error"], "Runtime (s)": r["runtime"]}
        for name, r in results.items()
    }).T
    print(summary.round(3))

## Key takeaways for practitioners

- **Same underlying idea, different APIs.** All three are in-context learners,
  but they don't share a common method signature — CausalPFN splits CATE/ATE
  into two estimator objects, Do-PFN folds treatment into the input and asks
  for two `do()` predictions, CausalFM takes a single 4-argument call with
  strict tensor shapes. Read the library's own quick-start before assuming
  another foundation model's calling convention carries over.
- **Install cost varies a lot.** Only CausalPFN is a plain `pip install`;
  Do-PFN and CausalFM require `git clone` plus manual `sys.path` wiring, and
  CausalFM additionally needs a checkpoint file — budget for that setup cost,
  it's one-time per environment.
- **"Fit" is not training.** No hyperparameters to tune, no train/val split to
  babysit — the context (your `X_train, T_train, Y_train`) *is* the input to a
  frozen network. If a model gives a bad answer, check the data going in
  (scaling, coding of `T`, covariate values outside its pretraining range)
  before looking for a "learning rate" to adjust — there isn't one.
- **Confounding is why we bothered simulating data.** The naive
  treated-vs-control gap printed in §1 is biased; the whole point of every
  method here (foundation model or classic metalearner) is to recover
  something closer to the true, confounder-adjusted effect from the same
  observational data.
- **Ground truth is a simulation-only luxury.** PEHE above only works because
  we know `tau_true`. On real data you only get an observed ATE to sanity-check
  against — CATE-level claims are unverifiable without ground truth, foundation
  model or not.
- **Hardware matters more than usual, and not for the reason you'd guess.**
  CausalPFN isn't "CUDA-only by design" — we tested directly and found it
  segfaults on Apple Silicon macOS for *both* CPU and MPS, most likely an
  unstable `scaled_dot_product_attention` kernel on that platform's backend,
  not a hard architectural requirement. It should run fine on Colab, GPU or
  CPU. Do-PFN/CausalFM ran fine on CPU here; expect fit+predict to just take
  longer without a GPU.